In [56]:
from pathlib import Path

VIDEO_IN = Path("Downloads/intput.mp4")   # <-- якщо файл називається інакше, зміни тут
print("Exists:", VIDEO_IN.exists())
print("Path:", VIDEO_IN.resolve())
print("Size (MB):", round(VIDEO_IN.stat().st_size / 1024 / 1024, 2) if VIDEO_IN.exists() else "N/A")

Exists: True
Path: C:\Users\refre\Downloads\intput.mp4
Size (MB): 239.54


In [57]:
!ffmpeg -y -ss 00:00:40 -to 00:01:20 -i "Downloads/input.mp4" -c copy "clip_1min.mp4"

ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with clang version 20.1.8
  configuration: --prefix=/c/miniconda3/conda-bld/ffmpeg_1762437060025/_h_env/Library --cc=clang.exe --ar=llvm-ar --nm=llvm-nm --ranlib=llvm-ranlib --strip= --disable-doc --enable-swresample --enable-swscale --enable-openssl --enable-libxml2 --enable-libtheora --enable-demuxer=dash --enable-postproc --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libdav1d --enable-zlib --enable-libaom --enable-pic --enable-shared --disable-static --disable-gpl --enable-version3 --disable-sdl2 --ld=lld-link --target-os=win64 --toolchain=msvc --host-cc=clang.exe --enable-cross-compile --disable-libmp3lame --host-extralibs= --disable-pthreads --enable-w32threads --extra-libs='ucrt.lib vcruntime.lib oldnames.lib' --disable-stripping
  libavutil      58. 29.100 / 58. 29.100
  libavcodec     60. 31.102 / 60. 31.102
  libavformat    60. 16.100 / 60. 16.100
  

In [58]:
import os
import cv2

video_path = "clip_1min.mp4"
out_folder = "frames_40_72"

os.makedirs(out_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
print("FPS:", fps)

i = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imwrite(os.path.join(out_folder, f"{i:06d}.jpg"), frame)
    i += 1

cap.release()
print("Saved frames:", i)

FPS: 29.97002997002997
Saved frames: 1200


In [60]:
import os
import cv2

# 1) Load frames (як у викладача)
folder = "frames_40_72"
frames = os.listdir(folder)
frames.sort()

# 2) Вибір стартового кадру
idx = 40   # можеш міняти, але зараз хай так як у тебе

# 3) Скільки кадрів трекати
# якщо хочеш ВСЮ хвилину — трекаємо до кінця папки
N = len(frames) - idx     # <- це “1 хвилина”, бо frames_1min це 1 хв

# 4) Перемикач трекера (як у викладача)
tracker_types = ["KCF", "CSRT"]
tracker_type = tracker_types[1]   # 0 = KCF, 1 = CSRT
# tracker_type = tracker_types[0] # <- розкоментуй це, щоб було CSRT

# 5) Read first frame
img0 = cv2.imread(os.path.join(folder, frames[idx]))
if img0 is None:
    raise RuntimeError("Не можу прочитати стартовий кадр. Перевір frames_40_72n.")

# 6) Select ROI (bbox) - готове OpenCV
roi = cv2.selectROI("Select ROI (ENTER) / Cancel (ESC)", img0, fromCenter=False, showCrosshair=True)
cv2.destroyWindow("Select ROI (ENTER) / Cancel (ESC)")

x, y, w, h = roi
print("ROI:", roi)
if w == 0 or h == 0:
    raise RuntimeError("ROI не вибрано. Виділи машину і натисни ENTER.")

# 7) Create tracker (KCF / CSRT)
def make_tracker(t):
    legacy = getattr(cv2, "legacy", None)

    if t == "KCF":
        if hasattr(cv2, "TrackerKCF_create"):
            return cv2.TrackerKCF_create()
        return legacy.TrackerKCF_create()

    if t == "CSRT":
        if hasattr(cv2, "TrackerCSRT_create"):
            return cv2.TrackerCSRT_create()
        return legacy.TrackerCSRT_create()

    raise RuntimeError("Невідомий tracker_type")

# 8) Run tracker for N frames (і зберігати)
out_dir = f"out_{tracker_type.lower()}"
os.makedirs(out_dir, exist_ok=True)

tracker = make_tracker(tracker_type)
ok = tracker.init(img0, (x, y, w, h))
if ok is False:
    raise RuntimeError(f"Не вдалося ініціалізувати {tracker_type}")

end = min(idx + N, len(frames))

for ii in range(idx, end):
    img = cv2.imread(os.path.join(folder, frames[ii]))
    if img is None:
        continue

    ok, bbox = tracker.update(img)

    if ok:
        bx, by, bw, bh = map(int, bbox)
        cv2.rectangle(img, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)
        cv2.putText(img, f"{tracker_type} OK", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    else:
        cv2.putText(img, f"{tracker_type} LOST", (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # SAVE (як у домашці)
    # щоб ffmpeg легко зібрав відео — зберігаємо з нумерацією кадрів:
    cv2.imwrite(os.path.join(out_dir, f"{ii:06d}.jpg"), img)

print(tracker_type, "saved frames to", out_dir, "| frames:", end - idx)

ROI: (494, 144, 81, 55)
CSRT saved frames to out_csrt | frames: 1160


In [45]:
import cv2

cap = cv2.VideoCapture("clip_1min.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("FPS:", fps)

FPS: 29.97002997002997


In [63]:
import shutil
from pathlib import Path

def renumber_to_seq(src_dir, dst_dir):
    src = Path(src_dir)
    dst = Path(dst_dir)

    if dst.exists():
        shutil.rmtree(dst)

    dst.mkdir(parents=True, exist_ok=True)

    imgs = sorted(src.glob("*.jpg"))
    for i, p in enumerate(imgs):
        shutil.copy2(p, dst / f"{i:06d}.jpg")

    print(dst_dir, "created:", len(imgs), "frames")

renumber_to_seq("out_kcf", "out_kcf_seq")
renumber_to_seq("out_csrt", "out_csrt_seq")

out_kcf_seq created: 1160 frames
out_csrt_seq created: 1160 frames


In [64]:
import cv2

cap = cv2.VideoCapture("clip_1min.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
print("FPS:", fps)

!ffmpeg -y -framerate {fps} -i out_kcf_seq/%06d.jpg  -c:v mpeg4 -q:v 3 -pix_fmt yuv420p kcf_result.mp4
!ffmpeg -y -framerate {fps} -i out_csrt_seq/%06d.jpg -c:v mpeg4 -q:v 3 -pix_fmt yuv420p csrt_result.mp4

FPS: 29.97002997002997


ffmpeg version 6.1.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with clang version 20.1.8
  configuration: --prefix=/c/miniconda3/conda-bld/ffmpeg_1762437060025/_h_env/Library --cc=clang.exe --ar=llvm-ar --nm=llvm-nm --ranlib=llvm-ranlib --strip= --disable-doc --enable-swresample --enable-swscale --enable-openssl --enable-libxml2 --enable-libtheora --enable-demuxer=dash --enable-postproc --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libdav1d --enable-zlib --enable-libaom --enable-pic --enable-shared --disable-static --disable-gpl --enable-version3 --disable-sdl2 --ld=lld-link --target-os=win64 --toolchain=msvc --host-cc=clang.exe --enable-cross-compile --disable-libmp3lame --host-extralibs= --disable-pthreads --enable-w32threads --extra-libs='ucrt.lib vcruntime.lib oldnames.lib' --disable-stripping
  libavutil      58. 29.100 / 58. 29.100
  libavcodec     60. 31.102 / 60. 31.102
  libavformat    60. 16.100 / 60. 16.100
  

In [ ]:
#Yes, there is a difference. CRT works better, its box adapts regardless of distance. It is more accurate but works slower. It handles difficult moments better if it simply holds the target better. I uploaded two videos to the images folder.